# Integração LoRa/Meshtastic, JSON e Alertas Ambientais

## Protótipo didático — trilha B: software-only / simulação

Este notebook demonstra uma cadeia local de dados ambientais: geração de leituras sintéticas, validação de JSON, cálculo de tendência, classificação de risco, mensagem curta de alerta e preparação de um envelope de integração simulado.

> **Limite importante:** não há hardware LoRa, sensores, GPS, nós Meshtastic, aplicativo móvel, broker MQTT ou transmissão de rádio nesta execução. Todos os dados usam `source: "simulation"`; as saídas validam a lógica de software, não uma comunicação física.

## Referências e distinções conceituais

- **LoRa** é a tecnologia de rádio de longo alcance e baixo consumo.
- **LoRaWAN** é uma arquitetura/protocolo LPWAN, normalmente em estrela, com nós, gateways e servidor de rede. O artigo principal usa essa arquitetura.
- **Meshtastic** usa LoRa em uma malha descentralizada; não é LoRaWAN e não deve ser descrito como tal.
- **MQTT/JSON** pode ser uma ponte IP entre borda e sistemas de dados; não substitui a comunicação LoRa nem demonstra transmissão pelo ar.

Fontes de estudo: [atividade do Período 8](../documentos_referencia/), [artigo LoRaWAN de 2023](https://doi.org/10.1016/j.iotcps.2023.04.005), [artigo Meshtastic de 2026](https://arxiv.org/abs/2605.20379), [MQTT no Meshtastic](https://meshtastic.org/docs/configuration/module/mqtt/) e [telemetria Meshtastic](https://meshtastic.org/docs/configuration/module/telemetry/).

A opção JSON do MQTT facilita integrações externas, mas a documentação do Meshtastic alerta que esses pacotes não são criptografados. Por isso, este exemplo não usa rede, credenciais, localização real ou broker público.

## Arquitetura da simulação

```text
[leituras sintéticas] -> [JSON canônico] -> [validador] -> [motor de risco]
                                                        |
                                                        v
[evidências locais] <- [exportador] <- [envelope MQTT simulado] <- [alerta curto]

Fronteira da simulação: nenhum pacote é codificado para rádio, enviado por LoRa,
publicado em MQTT ou exibido por um aplicativo Meshtastic.
```

A estrutura segue as quatro camadas discutidas na referência complementar: percepção (leitura), rede (contrato de mensagem), borda/processamento (validação e regra) e aplicação (alerta e evidência).

In [1]:
# Pré-requisitos: execute o notebook a partir da raiz do repositório.
from pathlib import Path
import sys

current_directory = Path.cwd().resolve()
PROJECT_ROOT = next((candidate for candidate in (current_directory, *current_directory.parents)
                     if (candidate / 'notebooks').is_dir() and (candidate / 'planejamento.md').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Não foi possível localizar a raiz do repositório a partir deste diretório.')

print(f'Python: {sys.version.split()[0]}')
print(f'Raiz do projeto: {PROJECT_ROOT}')
print('Modo de dados obrigatório: simulation')

Python: 3.14.4
Raiz do projeto: /home/mateus-tatsch/Documents/Americas TechGuard/Semana08/lora_meshtastic_integration_json_environmental_alerts_mobile_devices
Modo de dados obrigatório: simulation


In [2]:
# O pipeline usa somente a biblioteca padrão do Python.
from collections import Counter
from datetime import datetime
from typing import Any, Mapping
import json

OUTPUT_DIR = PROJECT_ROOT / 'outputs'
print('Imports e caminhos configurados.')

Imports e caminhos configurados.


In [3]:
# Configuração explícita e versionada da PoC.
SCHEMA_VERSION = '1.0'
SIMULATION_SOURCE = 'simulation'
SENSOR_TYPE = 'water_level'
SENSOR_UNIT = 'cm'
MAX_ALERT_LENGTH = 140
RISK_THRESHOLDS_CM = {
    'safe_upper': 50,
    'attention_upper': 100,
    'alert_upper': 150,
}

print('Faixas didáticas:', RISK_THRESHOLDS_CM)
print('Observação: os limiares não são calibração operacional de defesa civil.')

Faixas didáticas: {'safe_upper': 50, 'attention_upper': 100, 'alert_upper': 150}
Observação: os limiares não são calibração operacional de defesa civil.


## Contrato JSON

Cada entrada possui `schema_version`, `reading_id`, `device_id`, `timestamp`, `latitude`, `longitude`, `sensor_type`, `sensor_value`, `unit` e `source`. O processamento adiciona `trend_cm_per_min`, `risk_level`, `alert_message` e `processing_status`.

O contrato preserva os campos mínimos exigidos pela atividade. `risk_level` e `alert_message` são derivados pelo código, não recebidos como verdade de uma fonte externa. As coordenadas abaixo são sintéticas e servem somente para demonstrar formato e validação.

In [4]:
def build_synthetic_readings() -> list[dict[str, Any]]:
    """Cria leituras determinísticas; não representa dados de sensores reais."""
    common = {
        'schema_version': SCHEMA_VERSION,
        'device_id': 'sim-water-node-01',
        'node_name': 'Nó hídrico simulado 01',
        'latitude': -26.3000,
        'longitude': -48.8400,
        'altitude_m': None,
        'sensor_type': SENSOR_TYPE,
        'unit': SENSOR_UNIT,
        'source': SIMULATION_SOURCE,
    }
    observations = [
        ('sim-20260716-0001', '2026-07-16T12:00:00-03:00', 24.0),
        ('sim-20260716-0002', '2026-07-16T12:05:00-03:00', 50.0),
        ('sim-20260716-0003', '2026-07-16T12:10:00-03:00', 124.0),
        ('sim-20260716-0004', '2026-07-16T12:15:00-03:00', 152.0),
    ]
    return [
        {**common, 'reading_id': reading_id, 'timestamp': timestamp, 'sensor_value': level}
        for reading_id, timestamp, level in observations
    ]

synthetic_readings = build_synthetic_readings()
print(json.dumps(synthetic_readings, ensure_ascii=False, indent=2))

[
  {
    "schema_version": "1.0",
    "device_id": "sim-water-node-01",
    "node_name": "Nó hídrico simulado 01",
    "latitude": -26.3,
    "longitude": -48.84,
    "altitude_m": null,
    "sensor_type": "water_level",
    "unit": "cm",
    "source": "simulation",
    "reading_id": "sim-20260716-0001",
    "timestamp": "2026-07-16T12:00:00-03:00",
    "sensor_value": 24.0
  },
  {
    "schema_version": "1.0",
    "device_id": "sim-water-node-01",
    "node_name": "Nó hídrico simulado 01",
    "latitude": -26.3,
    "longitude": -48.84,
    "altitude_m": null,
    "sensor_type": "water_level",
    "unit": "cm",
    "source": "simulation",
    "reading_id": "sim-20260716-0002",
    "timestamp": "2026-07-16T12:05:00-03:00",
    "sensor_value": 50.0
  },
  {
    "schema_version": "1.0",
    "device_id": "sim-water-node-01",
    "node_name": "Nó hídrico simulado 01",
    "latitude": -26.3,
    "longitude": -48.84,
    "altitude_m": null,
    "sensor_type": "water_level",
    "unit": "cm"

In [5]:
REQUIRED_INPUT_FIELDS = {
    'schema_version', 'reading_id', 'device_id', 'timestamp', 'latitude', 'longitude',
    'sensor_type', 'sensor_value', 'unit', 'source'
}

def is_number(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool)

def validate_payload(payload: Mapping[str, Any]) -> list[dict[str, str]]:
    """Valida dados de entrada e devolve erros estruturados, sem lançar para o lote."""
    if not isinstance(payload, Mapping):
        return [{'field': 'payload', 'message': 'O payload precisa ser um objeto JSON.'}]

    errors: list[dict[str, str]] = []
    for field in sorted(REQUIRED_INPUT_FIELDS - set(payload)):
        errors.append({'field': field, 'message': 'Campo obrigatório ausente.'})

    if errors:
        return errors

    if payload['schema_version'] != SCHEMA_VERSION:
        errors.append({'field': 'schema_version', 'message': f'Versão esperada: {SCHEMA_VERSION}.'})
    for field in ('reading_id', 'device_id'):
        if not isinstance(payload[field], str) or not payload[field].strip():
            errors.append({'field': field, 'message': 'Deve ser texto não vazio.'})

    try:
        parsed_timestamp = datetime.fromisoformat(str(payload['timestamp']))
        if parsed_timestamp.tzinfo is None:
            errors.append({'field': 'timestamp', 'message': 'Use ISO 8601 com fuso horário.'})
    except ValueError:
        errors.append({'field': 'timestamp', 'message': 'Timestamp ISO 8601 inválido.'})

    latitude, longitude = payload['latitude'], payload['longitude']
    if not is_number(latitude) or not -90 <= latitude <= 90:
        errors.append({'field': 'latitude', 'message': 'Latitude deve estar entre -90 e 90.'})
    if not is_number(longitude) or not -180 <= longitude <= 180:
        errors.append({'field': 'longitude', 'message': 'Longitude deve estar entre -180 e 180.'})
    if payload['sensor_type'] != SENSOR_TYPE:
        errors.append({'field': 'sensor_type', 'message': f'Tipo aceito nesta PoC: {SENSOR_TYPE}.'})
    if not is_number(payload['sensor_value']) or payload['sensor_value'] < 0:
        errors.append({'field': 'sensor_value', 'message': 'Valor deve ser numérico e não negativo.'})
    if payload['unit'] != SENSOR_UNIT:
        errors.append({'field': 'unit', 'message': f'Unidade esperada: {SENSOR_UNIT}.'})
    if payload['source'] != SIMULATION_SOURCE:
        errors.append({'field': 'source', 'message': 'Esta execução aceita somente dados de simulation.'})
    return errors

valid_errors = validate_payload(synthetic_readings[0])
assert valid_errors == []
print('Payload válido: nenhuma inconsistência encontrada.')

Payload válido: nenhuma inconsistência encontrada.


In [6]:
# Casos inválidos são processados didaticamente; não interrompem o lote principal.
invalid_payloads = [
    {**synthetic_readings[0], 'reading_id': 'invalid-timestamp', 'timestamp': '16/07/2026 12:00'},
    {**synthetic_readings[0], 'reading_id': 'invalid-location', 'latitude': -91.0},
    {**synthetic_readings[0], 'reading_id': 'invalid-unit', 'unit': 'meters'},
]

for payload in invalid_payloads:
    print(f"{payload['reading_id']}: {validate_payload(payload)}")

invalid-timestamp: [{'field': 'timestamp', 'message': 'Timestamp ISO 8601 inválido.'}]
invalid-location: [{'field': 'latitude', 'message': 'Latitude deve estar entre -90 e 90.'}]
invalid-unit: [{'field': 'unit', 'message': 'Unidade esperada: cm.'}]


In [7]:
def parse_timestamp(payload: Mapping[str, Any]) -> datetime:
    return datetime.fromisoformat(str(payload['timestamp']))

def calculate_trend(previous: Mapping[str, Any] | None, current: Mapping[str, Any]) -> float | None:
    """Calcula a mudança de nível por minuto; a primeira leitura não tem tendência."""
    if previous is None:
        return None
    elapsed_minutes = (parse_timestamp(current) - parse_timestamp(previous)).total_seconds() / 60
    if elapsed_minutes <= 0:
        raise ValueError('O timestamp atual deve ser posterior ao anterior.')
    change = float(current['sensor_value']) - float(previous['sensor_value'])
    return round(change / elapsed_minutes, 2)

for index, reading in enumerate(synthetic_readings):
    previous = synthetic_readings[index - 1] if index else None
    print(reading['reading_id'], '->', calculate_trend(previous, reading), 'cm/min')

sim-20260716-0001 -> None cm/min
sim-20260716-0002 -> 5.2 cm/min
sim-20260716-0003 -> 14.8 cm/min
sim-20260716-0004 -> 5.6 cm/min


In [8]:
def classify_risk(water_level_cm: float) -> str:
    """Aplica a política didática baseada apenas no nível da água."""
    if water_level_cm < RISK_THRESHOLDS_CM['safe_upper']:
        return 'safe'
    if water_level_cm < RISK_THRESHOLDS_CM['attention_upper']:
        return 'attention'
    if water_level_cm < RISK_THRESHOLDS_CM['alert_upper']:
        return 'alert'
    return 'critical'

boundary_cases = {49: 'safe', 50: 'attention', 99: 'attention', 100: 'alert', 149: 'alert', 150: 'critical'}
for level, expected_risk in boundary_cases.items():
    actual_risk = classify_risk(level)
    assert actual_risk == expected_risk
    print(f'{level:>3} cm -> {actual_risk}')

print('Bordas de risco validadas.')

 49 cm -> safe
 50 cm -> attention
 99 cm -> attention
100 cm -> alert
149 cm -> alert
150 cm -> critical
Bordas de risco validadas.


In [9]:
def build_alert_message(water_level_cm: float, risk_level: str) -> str:
    """Gera texto curto; não substitui orientações oficiais de emergência."""
    level = f'{water_level_cm:.1f}'
    messages = {
        'safe': f'Nível seguro: {level} cm. Monitoramento simulado ativo.',
        'attention': f'ATENÇÃO: nível da água em {level} cm. Acompanhe os avisos oficiais.',
        'alert': f'ALERTA: nível da água em {level} cm. Prepare-se e acompanhe os avisos oficiais.',
        'critical': f'CRÍTICO: nível da água em {level} cm. Siga os avisos oficiais.',
    }
    message = messages[risk_level]
    if len(message) > MAX_ALERT_LENGTH:
        raise ValueError('Mensagem excede o limite didático configurado.')
    return message

for risk in ('safe', 'attention', 'alert', 'critical'):
    print(build_alert_message(124.0, risk))

Nível seguro: 124.0 cm. Monitoramento simulado ativo.
ATENÇÃO: nível da água em 124.0 cm. Acompanhe os avisos oficiais.
ALERTA: nível da água em 124.0 cm. Prepare-se e acompanhe os avisos oficiais.
CRÍTICO: nível da água em 124.0 cm. Siga os avisos oficiais.


In [10]:
def rejected_result(payload: Mapping[str, Any], errors: list[dict[str, str]]) -> dict[str, Any]:
    return {
        **dict(payload),
        'processing_status': 'rejected',
        'errors': errors,
    }

def process_reading(
    payload: Mapping[str, Any],
    previous_accepted: Mapping[str, Any] | None = None,
    seen_ids: set[str] | None = None,
) -> tuple[dict[str, Any], Mapping[str, Any] | None]:
    """Valida, enriquece e devolve também a última leitura aceita."""
    errors = validate_payload(payload)
    if errors:
        return rejected_result(payload, errors), previous_accepted
    if seen_ids is not None and payload['reading_id'] in seen_ids:
        return rejected_result(payload, [{'field': 'reading_id', 'message': 'Leitura duplicada no lote.'}]), previous_accepted

    try:
        trend = calculate_trend(previous_accepted, payload)
    except ValueError as error:
        return rejected_result(payload, [{'field': 'timestamp', 'message': str(error)}]), previous_accepted

    risk_level = classify_risk(float(payload['sensor_value']))
    result = {
        **dict(payload),
        'trend_cm_per_min': trend,
        'risk_level': risk_level,
        'alert_message': build_alert_message(float(payload['sensor_value']), risk_level),
        'processing_status': 'accepted',
    }
    return result, result

accepted_example, _ = process_reading(synthetic_readings[0])
rejected_example, _ = process_reading(invalid_payloads[0])
print('Aceito:', accepted_example['risk_level'], '| Rejeitado:', rejected_example['errors'][0]['field'])

Aceito: safe | Rejeitado: timestamp


In [11]:
def process_batch(payloads: list[Mapping[str, Any]]) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    results: list[dict[str, Any]] = []
    previous_accepted: Mapping[str, Any] | None = None
    seen_ids: set[str] = set()

    for payload in payloads:
        result, previous_accepted = process_reading(payload, previous_accepted, seen_ids)
        results.append(result)
        if result['processing_status'] == 'accepted':
            seen_ids.add(result['reading_id'])

    accepted = [item for item in results if item['processing_status'] == 'accepted']
    rejected = [item for item in results if item['processing_status'] == 'rejected']
    summary = {
        'simulation': True,
        'input_count': len(payloads),
        'accepted_count': len(accepted),
        'rejected_count': len(rejected),
        'risk_counts': dict(Counter(item['risk_level'] for item in accepted)),
    }
    return results, summary

processed_results, batch_summary = process_batch(synthetic_readings)
assert batch_summary['accepted_count'] == 4
assert batch_summary['risk_counts'] == {'safe': 1, 'attention': 1, 'alert': 1, 'critical': 1}
print(json.dumps(batch_summary, ensure_ascii=False, indent=2))

{
  "simulation": true,
  "input_count": 4,
  "accepted_count": 4,
  "rejected_count": 0,
  "risk_counts": {
    "safe": 1,
    "attention": 1,
    "alert": 1,
    "critical": 1
  }
}


In [12]:
def to_meshtastic_mqtt_simulation(processed_payload: Mapping[str, Any]) -> dict[str, Any]:
    """Cria contrato ilustrativo de borda; não abre conexão e não codifica pacote de rádio."""
    if processed_payload['processing_status'] != 'accepted':
        raise ValueError('Somente leituras aceitas podem ser preparadas para integração simulada.')
    return {
        'simulation': True,
        'transport': 'MQTT JSON bridge illustration — not a Meshtastic radio packet',
        'topic': f'americas-techguard/simulation/{processed_payload["device_id"]}/telemetry',
        'payload': dict(processed_payload),
        'warning': 'Nenhuma publicação MQTT, transmissão LoRa ou notificação móvel ocorreu.',
    }

alert_example = next(item for item in processed_results if item['risk_level'] == 'alert')
simulated_envelope = to_meshtastic_mqtt_simulation(alert_example)
print(json.dumps(simulated_envelope, ensure_ascii=False, indent=2))

{
  "simulation": true,
  "transport": "MQTT JSON bridge illustration — not a Meshtastic radio packet",
  "topic": "americas-techguard/simulation/sim-water-node-01/telemetry",
  "payload": {
    "schema_version": "1.0",
    "device_id": "sim-water-node-01",
    "node_name": "Nó hídrico simulado 01",
    "latitude": -26.3,
    "longitude": -48.84,
    "altitude_m": null,
    "sensor_type": "water_level",
    "unit": "cm",
    "source": "simulation",
    "reading_id": "sim-20260716-0003",
    "timestamp": "2026-07-16T12:10:00-03:00",
    "sensor_value": 124.0,
    "trend_cm_per_min": 14.8,
    "risk_level": "alert",
    "alert_message": "ALERTA: nível da água em 124.0 cm. Prepare-se e acompanhe os avisos oficiais.",
    "processing_status": "accepted"
  },
  "warning": "Nenhuma publicação MQTT, transmissão LoRa ou notificação móvel ocorreu."
}


## Limite do adaptador simulado

O envelope anterior representa uma possível mensagem entregue a uma camada de borda depois de uma integração real. Ele **não** é um pacote de rádio Meshtastic, não representa protobuf, não calcula Time-on-Air e não executa *managed flooding*. Em uma futura etapa física, o adaptador deverá ser substituído por integração documentada com dispositivos reais, canal configurado, firmware compatível e broker protegido.

In [13]:
def print_results_table(rows: list[Mapping[str, Any]]) -> None:
    columns = [
        ('reading_id', 'Leitura'),
        ('sensor_value', 'Nível (cm)'),
        ('trend_cm_per_min', 'Tendência'),
        ('risk_level', 'Risco'),
        ('processing_status', 'Estado'),
    ]
    rendered = []
    for row in rows:
        rendered.append([str(row.get(key, '')) for key, _ in columns])
    widths = [max(len(label), *(len(row[index]) for row in rendered)) for index, (_, label) in enumerate(columns)]
    header = ' | '.join(label.ljust(widths[index]) for index, (_, label) in enumerate(columns))
    separator = '-+-'.join('-' * width for width in widths)
    print(header)
    print(separator)
    for row in rendered:
        print(' | '.join(value.ljust(widths[index]) for index, value in enumerate(row)))

print_results_table(processed_results)
print('\nMensagens de alerta:')
for result in processed_results:
    print(f"- [{result['risk_level']}] {result['alert_message']}")

Leitura           | Nível (cm) | Tendência | Risco     | Estado  
------------------+------------+-----------+-----------+---------
sim-20260716-0001 | 24.0       | None      | safe      | accepted
sim-20260716-0002 | 50.0       | 5.2       | attention | accepted
sim-20260716-0003 | 124.0      | 14.8      | alert     | accepted
sim-20260716-0004 | 152.0      | 5.6       | critical  | accepted

Mensagens de alerta:
- [safe] Nível seguro: 24.0 cm. Monitoramento simulado ativo.
- [attention] ATENÇÃO: nível da água em 50.0 cm. Acompanhe os avisos oficiais.
- [alert] ALERTA: nível da água em 124.0 cm. Prepare-se e acompanhe os avisos oficiais.
- [critical] CRÍTICO: nível da água em 152.0 cm. Siga os avisos oficiais.


In [14]:
def write_json(path: Path, content: Any) -> Path:
    path.write_text(json.dumps(content, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return path

OUTPUT_DIR.mkdir(exist_ok=True)
rejected_results = [process_reading(payload)[0] for payload in invalid_payloads]
output_files = {
    'processed': write_json(OUTPUT_DIR / 'processed_readings.json', processed_results),
    'rejected': write_json(OUTPUT_DIR / 'rejected_readings.json', rejected_results),
    'summary': write_json(OUTPUT_DIR / 'execution_summary.json', batch_summary),
    'envelope': write_json(OUTPUT_DIR / 'meshtastic_mqtt_envelope_simulated.json', simulated_envelope),
}
for label, path in output_files.items():
    print(f'{label}: {path.relative_to(PROJECT_ROOT)}')

processed: outputs/processed_readings.json
rejected: outputs/rejected_readings.json
summary: outputs/execution_summary.json
envelope: outputs/meshtastic_mqtt_envelope_simulated.json


In [15]:
# Verificação de persistência: relê evidências gravadas pelo notebook.
loaded_processed = json.loads(output_files['processed'].read_text(encoding='utf-8'))
loaded_rejected = json.loads(output_files['rejected'].read_text(encoding='utf-8'))
loaded_summary = json.loads(output_files['summary'].read_text(encoding='utf-8'))

assert len(loaded_processed) == loaded_summary['accepted_count']
assert len(loaded_rejected) == len(invalid_payloads)
assert {item['risk_level'] for item in loaded_processed} == {'safe', 'attention', 'alert', 'critical'}
assert all(item['source'] == SIMULATION_SOURCE for item in loaded_processed)
print('Evidências relidas com sucesso.')
print('Contagens:', loaded_summary)

Evidências relidas com sucesso.
Contagens: {'simulation': True, 'input_count': 4, 'accepted_count': 4, 'rejected_count': 0, 'risk_counts': {'safe': 1, 'attention': 1, 'alert': 1, 'critical': 1}}


## Mapeamento para uma etapa física futura

Com acesso autorizado em Joinville, o contrato acima pode ser reutilizado com uma T-Beam V1.1, Heltec LoRa 32 V3 ou TTGO LoRa T3S3 1.2 configurada para 915 MHz. A sequência mínima seria: confirmar a revisão do rádio e compatibilidade do firmware; usar pelo menos dois nós (`CLIENT` e `ROUTER`/`GATEWAY`); instalar antena adequada antes de transmitir; validar serial e BLE; calibrar o sensor; e somente então medir RSSI, SNR, PDR e atraso.

A revisão do hardware é essencial: variantes de placas podem usar transceptores diferentes. O sensor de nível também não deve ser assumido como telemetria nativa sem conferir suporte do firmware. Em uma integração MQTT real, utilizar broker próprio, TLS e credenciais fora do Git.

## Resultado, limitações e próximos passos

Esta PoC valida que leituras sintéticas podem ser estruturadas em JSON, rejeitadas quando inválidas, enriquecidas com tendência e risco, transformadas em mensagem curta e exportadas como evidência. Ela apoia o Americas TechGuard como uma camada intermediária entre futura instrumentação ambiental, processamento, dashboards e disseminação de alertas.

Não foram validados: precisão do sensor, GPS, autonomia, rádio, alcance, malha, interferência, MQTT, segurança de transporte, aplicativo móvel, entendimento do alerta por usuários ou limiares operacionais. Os próximos passos são testar os mesmos contratos com hardware autorizado, medir o enlace em campo e definir regras de risco com dados e responsáveis competentes.